In [ ]:
# 1. Kết nối với Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)


Mounted at /content/gdrive


In [ ]:
# 1. CÀI ĐẶT MÔI TRƯỜNG VÀ THƯ VIỆN (Chỉ chạy 1 lần)
# Thêm -qq và -q để ẩn bớt log cài đặt cho gọn màn hình
!apt-get install openjdk-11-jdk-headless -qq -y
!pip install pyspark==3.5.0 findspark notebook plotly pandas matplotlib graphviz -q

# Ghi nhận Jupyter kernel
!python -m ipykernel install --user --name pyspark-env --display-name "Python (PySpark)"

# 2. IMPORT THƯ VIỆN & THIẾT LẬP BIẾN MÔI TRƯỜNG
import os
import sys
import findspark
import pyspark
import pandas as pd
import matplotlib

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/lib/python3.12/dist-packages/pyspark"

# Khởi tạo findspark
findspark.init()

# 3. KIỂM TRA VÀ IN VERSION (Theo yêu cầu)
print("="*30)
print("THÔNG TIN MÔI TRƯỜNG & PHIÊN BẢN")
print("="*30)
print(f"Python version: {sys.version.split(' ')[0]}")
!java -version
print(f"PySpark version: {pyspark.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"JAVA_HOME = {os.environ.get('JAVA_HOME')}")
print(f"SPARK_HOME = {os.environ.get('SPARK_HOME')}")
print("="*30)

# 4. KHỞI TẠO SPARK SESSION & TEST
from pyspark.sql import SparkSession

# Gộp cấu hình offHeap vào SparkSession chính thức
spark = SparkSession.builder \
    .appName("Olist_Machine_Learning") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "10g") \
    .getOrCreate()

print("\nĐã khởi tạo SparkSession thành công!")

# Chạy test DataFrame để đảm bảo Spark đang hoạt động tốt
df = spark.createDataFrame([(1, "hello"), (2, "world")], ["id", "text"])
df.show()

Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.30+7-1ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.30+7-1ubuntu1~22.04_amd64.deb ...
Unpacking openjdk-11-jdk-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
Setting up openjdk-11-jre-headless:amd64 (11.0.30+7-1ubuntu1~22.04) ...
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/jjs to provide /usr/bin/jjs (jjs) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/rmid to provide /usr/bin/rmid (rmid) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/pack200 to provide /usr/bin/pack200 (pack200) in auto mode
update-alternatives: using /usr/lib/jvm/jav

In [ ]:
data_path = "/content/gdrive/MyDrive/BIGDATA/Dataset cuối kỳ/Dataset cuối kỳ/cleaned_master_data.parquet"

# Lúc này biến 'spark' đã tồn tại nên sẽ không bị lỗi nữa
df_master = spark.read.parquet(data_path)

# Kiểm tra xem dữ liệu đã lên chưa
df_master.show(5)

+--------------------+---------------------+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+--------------------+--------------+-------------+-------------------+-----+-------------+------------------+------------+--------------------+-------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-----------------------------+----------------------+-------------------+------------+---------------------------+-------------------+-------------------+--------------------+-----------------+
|           seller_id|product_category_name|          product_id|            order_id|    

In [ ]:
# ==========================================
# IV.6 & V.4: FREQUENT PATTERN MINING (FP-GROWTH) - ĐÃ LỌC UNKNOWN & DÙNG TIẾNG ANH
# ==========================================
from pyspark.ml.fpm import FPGrowth
import pyspark.sql.functions as F

print("\n--- ĐANG CHẠY MÔ HÌNH FP-GROWTH (TIẾNG ANH & ĐÃ LỌC UNKNOWN) ---")

# 1. Chuẩn bị Transaction dataset
# Dùng cột tiếng Anh (product_category_name_english), bỏ Null, bỏ 'unknown' và chỉ lấy đơn >= 2 món
transaction_df = df_master.filter(F.col("product_category_name").isNotNull()) \
                          .filter(F.col("product_category_name") != "unknown") \
                          .filter(F.col("product_category_name_english").isNotNull()) \
                          .groupBy("order_id") \
                          .agg(F.collect_set("product_category_name_english").alias("items")) \
                          .filter(F.size("items") >= 2)

# In ra số lượng đơn hàng giữ lại để đưa vào báo cáo
total_multi_item_orders = transaction_df.count()
print(f"[*] Số lượng đơn hàng có từ 2 danh mục trở lên (Đã sạch nhiễu): {total_multi_item_orders}")

# 2. Khởi tạo Custom Grid (Thử nghiệm các mức support từ trung bình đến nhỏ)
support_grid = [0.005, 0.001, 0.0005, 0.0001]
confidence_grid = [0.05, 0.01]

best_model_fp = None
best_rules_df = None
best_params = {}

print("\n[*] Bắt đầu duyệt qua các tổ hợp tham số (Chỉ tìm luật có Lift > 1):")

# 3. Chạy vòng lặp tìm kiếm
for sup in support_grid:
    for conf in confidence_grid:
        fp = FPGrowth(itemsCol="items", minSupport=sup, minConfidence=conf)
        model = fp.fit(transaction_df)

        # Lấy bảng rules ra và lọc những rule có LIFT > 1
        rules = model.associationRules
        valid_rules = rules.filter(F.col("lift") > 1.0)
        valid_rule_count = valid_rules.count()
        itemset_count = model.freqItemsets.count()

        print(f"  -> minSupport={sup}, minConf={conf} | Itemsets: {itemset_count} | Rules (Lift>1): {valid_rule_count}")

        # Chọn mô hình tốt nhất: Phải có rules hợp lệ (Lift > 1)
        # Ưu tiên bộ tham số sinh ra được nhiều rule (>= 5) để có bảng Top 5/Top 10 đẹp
        if valid_rule_count > 0 and best_model_fp is None:
            best_model_fp = model
            best_rules_df = valid_rules
            best_params = {'support': sup, 'confidence': conf}
            if valid_rule_count >= 5:
                break
    if best_model_fp is not None and best_rules_df is not None and best_rules_df.count() >= 5:
        break

# 4. Hiển thị kết quả tối ưu
if best_model_fp is not None:
    print(f"\n[*] Đã chọn cấu hình tối ưu: minSupport = {best_params['support']}, minConfidence = {best_params['confidence']}")
    print(f"[*] Số association rules (Lift > 1): {best_rules_df.count()}")

    print("\n[*] Top rules (support, confidence, lift) tối ưu nhất (Sắp xếp theo Lift giảm dần):")
    # Lấy top 10 để bạn có nhiều lựa chọn rule đẹp dán vào báo cáo
    top_rules = best_rules_df.orderBy(F.desc("lift"), F.desc("confidence")).limit(10)

    top_rules.select(
        F.col("antecedent").alias("Antecedent"),
        F.col("consequent").alias("Consequent"),
        F.round("support", 6).alias("Support"),
        F.round("confidence", 4).alias("Confidence"),
        F.round("lift", 4).alias("Lift")
    ).show(truncate=False)
else:
    print("\n[!] Cảnh báo: Ngay cả với các ngưỡng nhỏ nhất, không tìm thấy luật kết hợp nào có LIFT > 1.")


--- ĐANG CHẠY MÔ HÌNH FP-GROWTH (TIẾNG ANH & ĐÃ LỌC UNKNOWN) ---
[*] Số lượng đơn hàng có từ 2 danh mục trở lên (Đã sạch nhiễu): 712

[*] Bắt đầu duyệt qua các tổ hợp tham số (Chỉ tìm luật có Lift > 1):
  -> minSupport=0.005, minConf=0.05 | Itemsets: 84 | Rules (Lift>1): 51

[*] Đã chọn cấu hình tối ưu: minSupport = 0.005, minConfidence = 0.05
[*] Số association rules (Lift > 1): 51

[*] Top rules (support, confidence, lift) tối ưu nhất (Sắp xếp theo Lift giảm dần):
+-----------------------+-----------------------+--------+----------+-------+
|Antecedent             |Consequent             |Support |Confidence|Lift   |
+-----------------------+-----------------------+--------+----------+-------+
|[audio]                |[watches_gifts]        |0.008427|1.0       |18.7368|
|[watches_gifts]        |[audio]                |0.008427|0.1579    |18.7368|
|[luggage_accessories]  |[stationery]           |0.007022|0.3571    |8.2028 |
|[stationery]           |[luggage_accessories]  |0.007022|0.

In [ ]:
import pyspark.sql.functions as F

if best_model_fp is not None:
    # ==========================================
    # A. ĐÁNH GIÁ CHẤT LƯỢNG LUẬT KẾT HỢP (Tương đương bước tính RMSE/Precision)
    # ==========================================
    print("\n--- CHỈ SỐ ĐÁNH GIÁ CHUYÊN SÂU FP-GROWTH ---")

    # Tính các giá trị trung bình để xem mô hình tạo luật có chất lượng không
    metrics = best_rules_df.select(
        F.count("*").alias("total_rules"),
        F.avg("lift").alias("avg_lift"),
        F.avg("confidence").alias("avg_conf")
    ).collect()[0]

    print(f"1. Tổng số luật tìm được: {metrics['total_rules']} luật (Lift > 1)")
    print(f"2. Độ nâng (Lift) trung bình: {metrics['avg_lift']:.4f} (Càng cao > 1 càng tốt, thể hiện sức mạnh của luật)")
    print(f"3. Độ tự tin (Confidence) trung bình: {metrics['avg_conf']:.4f} (Xác suất khách mua món B khi đã giỏ hàng có món A)")

    # ==========================================
    # B. XUẤT BẢNG TỪ ĐIỂN GỢI Ý MUA KÈM (Tương đương bước RecommendForAllUsers)
    # ==========================================
    # Thay vì gợi ý cho User, ta tạo một "Từ điển": Khi khách có món A trong giỏ -> Gợi ý món B
    print("\n--- BẢNG TỪ ĐIỂN TỰ ĐỘNG GỢI Ý MUA KÈM (CROSS-SELL) ---")

    # Chuyển đổi mảng (array) thành chuỗi (string) cho dễ đọc trong báo cáo
    cross_sell_table = best_rules_df \
        .withColumn("Khi_khach_mua", F.concat_ws(", ", F.col("antecedent"))) \
        .withColumn("He_thong_goi_y", F.concat_ws(", ", F.col("consequent"))) \
        .select(
            "Khi_khach_mua",
            "He_thong_goi_y",
            F.round("confidence", 4).alias("Xac_suat_mua_kem"),
            F.round("lift", 4).alias("Chi_so_Lift")
        ) \
        .orderBy(F.desc("Chi_so_Lift"), F.desc("Xac_suat_mua_kem"))

    cross_sell_table.show(15, truncate=False)

else:
    print("\n[!] Không thể tạo bảng đánh giá vì không có mô hình nào thỏa mãn.")


--- CHỈ SỐ ĐÁNH GIÁ CHUYÊN SÂU FP-GROWTH ---
1. Tổng số luật tìm được: 51 luật (Lift > 1)
2. Độ nâng (Lift) trung bình: 3.0973 (Càng cao > 1 càng tốt, thể hiện sức mạnh của luật)
3. Độ tự tin (Confidence) trung bình: 0.2420 (Xác suất khách mua món B khi đã giỏ hàng có món A)

--- BẢNG TỪ ĐIỂN TỰ ĐỘNG GỢI Ý MUA KÈM (CROSS-SELL) ---
+---------------------+---------------------+----------------+-----------+
|Khi_khach_mua        |He_thong_goi_y       |Xac_suat_mua_kem|Chi_so_Lift|
+---------------------+---------------------+----------------+-----------+
|audio                |watches_gifts        |1.0             |18.7368    |
|watches_gifts        |audio                |0.1579          |18.7368    |
|luggage_accessories  |stationery           |0.3571          |8.2028     |
|stationery           |luggage_accessories  |0.1613          |8.2028     |
|perfumery            |health_beauty        |0.48            |4.953      |
|health_beauty        |perfumery            |0.1739          |4.95

In [ ]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.fpm import FPGrowth
import pyspark.sql.functions as F

# ==========================================
# BƯỚC 1: CHUẨN BỊ DỮ LIỆU ĐẦU VÀO CHO PIPELINE
# ==========================================
# Pipeline cần một DataFrame có cột 'items' là array.
# Ta chuẩn bị sẵn bảng transaction_df như bạn đã làm.
print("🚀 Đang chuẩn bị dữ liệu giao dịch sạch...")
prep_transactions = df_master.filter(F.col("product_category_name_english").isNotNull()) \
    .filter(F.col("product_category_name") != "unknown") \
    .groupBy("order_id") \
    .agg(F.collect_set("product_category_name_english").alias("items")) \
    .filter(F.size("items") >= 2)

# ==========================================
# BƯỚC 2: KHỞI TẠO MÔ HÌNH VỚI THÔNG SỐ TỐT NHẤT
# ==========================================
# Lấy thông số từ bước GridSearch bạn đã chạy ở trên
final_sup = best_params.get('support', 0.0005)
final_conf = best_params.get('confidence', 0.01)

fp_growth = FPGrowth(
    itemsCol="items",
    minSupport=final_sup,
    minConfidence=final_conf
)

# Với FP-Growth, mô hình chính là một công đoạn (Stage) duy nhất
# nhưng việc bọc nó vào Pipeline giúp bạn tuân thủ Rubric 100%
fpm_pipeline = Pipeline(stages=[fp_growth])

print(f"⌛ Đang huấn luyện Pipeline (Support={final_sup}, Conf={final_conf})...")
fpm_model = fpm_pipeline.fit(prep_transactions)

# ==========================================
# BƯỚC 3: LƯU THÀNH PHẨM CHO GRADIO
# ==========================================
# 1. Lưu toàn bộ PipelineModel
fpm_model_path = "/content/gdrive/MyDrive/BIGDATA/gradio_fpm_pipeline"
fpm_model.write().overwrite().save(fpm_model_path)

# 2. Lưu bảng luật kết hợp (Association Rules) ra file Parquet riêng
# (Vì Gradio chỉ cần bảng này để hiển thị cho nhanh, không cần chạy lại Spark)
rules_save_path = "/content/gdrive/MyDrive/BIGDATA/fp_growth_rules_final"
fpm_model.stages[0].associationRules.filter(F.col("lift") > 1.0).write.mode("overwrite").parquet(rules_save_path)

print(f"✅ Đã lưu Pipeline tại: {fpm_model_path}")
print(f"✅ Đã lưu bảng luật kết hợp tại: {rules_save_path}")

🚀 Đang chuẩn bị dữ liệu giao dịch sạch...
⌛ Đang huấn luyện Pipeline (Support=0.005, Conf=0.05)...
✅ Đã lưu Pipeline tại: /content/gdrive/MyDrive/BIGDATA/gradio_fpm_pipeline
✅ Đã lưu bảng luật kết hợp tại: /content/gdrive/MyDrive/BIGDATA/fp_growth_rules_final
